MCP AI Client

In [ ]:
%run ~/data/env.py
! cat ~/data/env.py

In [ ]:
from openai import OpenAI
from mcp.client.sse import sse_client
from mcp import ClientSession

import json
import os

MCP_URL = "http://localhost:8000/sse"

client = OpenAI(
    api_key=OPENAI_API_KEY,
    base_url=AI_BASE_URL,
)

AI Test

In [ ]:
response = client.responses.create(
    model=AI_MODEL,
    input="Which car brands sold the most units in 2021?"
)

print(response.output_text)

Jetzt mit MCP-Tool-Anbindung. Für das Filesystem-Beispiel definieren wir ein OpenAI-Tool filesystem_cli. Das Modell kann damit erlaubte MCP-Tools wie pwd, ls, df, uname, cat oder grep anfordern.

In [ ]:
tools = [
    {
        "type": "function",
        "name": "filesystem_cli",
        "description": "Ruft erlaubte Filesystem- und Linux-CLI-Tools über den lokalen MCP-Server auf.",
        "parameters": {
            "type": "object",
            "properties": {
                "tool": {
                    "type": "string",
                    "description": "Name des MCP-Tools.",
                    "enum": ["pwd", "ls", "cat", "grep", "df", "uname"]
                },
                "arguments": {
                    "type": "object",
                    "description": "Argumente für das MCP-Tool."
                }
            },
            "required": ["tool", "arguments"],
            "additionalProperties": False
        }
    }
]

MCP-Bridge-Funktion:

In [ ]:
async def call_mcp_tool(tool_name: str, arguments: dict | None = None):
    async with sse_client(MCP_URL) as (read, write):
        async with ClientSession(read, write) as session:
            await session.initialize()
            return await session.call_tool(tool_name, arguments or {})

Hilfsfunktion, um MCP-Resultate in Text zu wandeln:

In [ ]:
def mcp_result_to_text(result) -> str:
    if hasattr(result, "content"):
        parts = []
        for item in result.content:
            if hasattr(item, "text"):
                parts.append(item.text)
            else:
                parts.append(str(item))
        return "\n\n".join(parts)

    if hasattr(result, "contents"):
        parts = []
        for item in result.contents:
            if hasattr(item, "text"):
                parts.append(item.text)
            else:
                parts.append(str(item))
        return "\n\n".join(parts)

    return str(result)

Jetzt der eigentliche OpenAI-Loop:

In [ ]:
async def ask_with_mcp(prompt: str):
    response = client.responses.create(
        model=AI_MODEL,
        input=prompt,
        tools=tools,
    )

    # Falls kein Tool gebraucht wurde, direkt Text ausgeben
    if not response.output:
        return response.output_text

    tool_outputs = []

    for item in response.output:
        if item.type != "function_call":
            continue

        if item.name != "filesystem_cli":
            continue

        args = json.loads(item.arguments)

        mcp_tool = args["tool"]
        mcp_arguments = args.get("arguments", {})

        mcp_result = await call_mcp_tool(mcp_tool, mcp_arguments)
        mcp_text = mcp_result_to_text(mcp_result)

        tool_outputs.append({
            "type": "function_call_output",
            "call_id": item.call_id,
            "output": mcp_text,
        })

    # Falls Tool Calls vorhanden waren: Resultate zurück ans Modell geben
    if tool_outputs:
        response = client.responses.create(
            model=AI_MODEL,
            previous_response_id=response.id,
            input=tool_outputs,
        )

    return response.output_text

Verwendung:

In [ ]:
answer = await ask_with_mcp("Zeige mir das aktuelle Arbeitsverzeichnis.")
print(answer)